In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

ROOT = "/content/drive/MyDrive/RoofMaterialProject"

df_5class = pd.read_csv(
    f"{ROOT}/features/roof_features_44_with_labels.csv"
)

print("Dataset shape:", df_5class.shape)
print("\nClass distribution:")
print(df_5class["roof_material"].value_counts())

Dataset shape: (10356, 50)

Class distribution:
roof_material
healthy_metal      5544
irregular_metal    3632
incomplete          660
concrete_cement     497
other                23
Name: count, dtype: int64


In [3]:
feature_cols = [
    c for c in df_5class.columns
    if c.startswith(("r_", "g_", "b_", "h_", "s_", "v_", "lbp_", "glcm_"))
]

X_5 = df_5class[feature_cols].astype(np.float32)
y_5 = df_5class["roof_material"]

print("Number of features:", len(feature_cols))
print("Number of classes:", y_5.nunique())
print("Total samples:", len(y_5))

Number of features: 44
Number of classes: 5
Total samples: 10356


In [4]:
from sklearn.model_selection import train_test_split

X_train_5, X_val_5, y_train_5, y_val_5 = train_test_split(
    X_5,
    y_5,
    test_size=0.20,
    random_state=42,
    stratify=y_5
)

print("Training samples:", len(X_train_5))
print("Validation samples:", len(X_val_5))

print("\nTraining distribution:")
print(y_train_5.value_counts())

print("\nValidation distribution:")
print(y_val_5.value_counts())

Training samples: 8284
Validation samples: 2072

Training distribution:
roof_material
healthy_metal      4435
irregular_metal    2905
incomplete          528
concrete_cement     398
other                18
Name: count, dtype: int64

Validation distribution:
roof_material
healthy_metal      1109
irregular_metal     727
incomplete          132
concrete_cement      99
other                 5
Name: count, dtype: int64


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

svm_5class = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        class_weight="balanced"
    ))
])

svm_5class.fit(X_train_5, y_train_5)

print("5-class SVM training completed!")

5-class SVM training completed!


In [6]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_pred_5 = svm_5class.predict(X_val_5)

print("Accuracy:", accuracy_score(y_val_5, y_pred_5))

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_val_5, y_pred_5)
)

print(
    "Macro-F1:",
    f1_score(y_val_5, y_pred_5, average="macro")
)

print("\nClassification Report:")
print(classification_report(y_val_5, y_pred_5, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_5, y_pred_5))

Accuracy: 0.6230694980694981
Balanced Accuracy: 0.4310594155405211
Macro-F1: 0.39127069336907516

Classification Report:
                 precision    recall  f1-score   support

concrete_cement       0.19      0.42      0.27        99
  healthy_metal       0.81      0.73      0.77      1109
     incomplete       0.31      0.48      0.38       132
irregular_metal       0.58      0.52      0.55       727
          other       0.00      0.00      0.00         5

       accuracy                           0.62      2072
      macro avg       0.38      0.43      0.39      2072
   weighted avg       0.67      0.62      0.64      2072


Confusion Matrix:
[[ 42  16  15  26   0]
 [ 59 810  27 212   1]
 [ 23  11  64  34   0]
 [ 90 159 103 375   0]
 [  2   2   0   1   0]]


In [8]:
import pandas as pd
import os

ROOT = "/content/drive/MyDrive/RoofMaterialProject"
save_path = os.path.join(
    ROOT, "features", "svm_5class_validation_predictions.csv"
)

predictions_df = pd.DataFrame({
    "actual": y_val_5.values,
    "predicted": y_pred_5
})

predictions_df.to_csv(save_path, index=False)

print("Predictions saved successfully!")
print("Location:", save_path)
print("\nFirst 5 predictions:")
display(predictions_df.head())

Predictions saved successfully!
Location: /content/drive/MyDrive/RoofMaterialProject/features/svm_5class_validation_predictions.csv

First 5 predictions:


,actual,predicted
0,irregular_metal,healthy_metal
1,irregular_metal,healthy_metal
2,irregular_metal,concrete_cement
3,healthy_metal,healthy_metal
4,healthy_metal,healthy_metal


In [7]:
import joblib

joblib.dump(
    svm_5class,
    f"{ROOT}/features/svm_5class_baseline.joblib"
)

print("5-class SVM saved successfully!")

5-class SVM saved successfully!


In [12]:
"""investigation on the others class"""

'investigation on the others class'

In [10]:
other_samples = df_5class[
    df_5class["roof_material"] == "other"
]

print("Other samples:", len(other_samples))

print("\nRegion distribution:")
print(other_samples["region"].value_counts())

print("\nAvailable metadata:")
display(other_samples.head(23))

Other samples: 23

Region distribution:
region
borde_soacha    15
borde_rural      8
Name: count, dtype: int64

Available metadata:


,region,building_id,roof_material,patch_path,status,error,r_mean,r_std,r_p10,r_p90,...,glcm_homogeneity_d2_mean,glcm_homogeneity_d2_std,glcm_energy_d1_mean,glcm_energy_d1_std,glcm_energy_d2_mean,glcm_energy_d2_std,glcm_correlation_d1_mean,glcm_correlation_d1_std,glcm_correlation_d2_mean,glcm_correlation_d2_std
1008,borde_rural,7a391026,other,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,137.370670,38.862300,93.0,191.00000,...,0.780313,0.033173,0.258504,0.014140,0.239923,0.011184,0.960460,0.010293,0.938617,0.017589
1244,borde_rural,7a3d7dfa,other,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,128.775270,50.599403,57.0,186.00000,...,0.673763,0.035730,0.188175,0.010826,0.171807,0.009219,0.942529,0.017353,0.907024,0.032308
1573,borde_rural,7a1e6f46,other,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,90.531860,49.800728,25.0,165.00000,...,0.812707,0.018521,0.278951,0.009362,0.263321,0.007676,0.973375,0.006514,0.959065,0.008335
1791,borde_rural,7a3dd502,other,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,145.833440,40.895950,84.0,181.00000,...,0.737133,0.030995,0.286707,0.010257,0.267959,0.011058,0.937769,0.015370,0.899690,0.027242
1829,borde_rural,7a3f2984,other,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,156.297850,54.573616,61.0,215.00000,...,0.695972,0.026487,0.208029,0.010583,0.191966,0.007891,0.928662,0.018844,0.890277,0.026063
2132,borde_rural,7a1dc41a,other,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,123.762510,54.214474,29.0,180.00000,...,0.699254,0.031918,0.217845,0.009607,0.202588,0.009455,0.954708,0.012110,0.928140,0.016410
3124,borde_rural,7a255400,other,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,110.876830,47.883670,46.0,172.00000,...,0.678284,0.038375,0.184995,0.007666,0.168413,0.010102,0.940509,0.014420,0.904508,0.022540
3523,borde_rural,7a343d44,other,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,153.264530,59.870213,61.0,238.00000,...,0.732481,0.021495,0.223035,0.007660,0.211111,0.005860,0.951856,0.013203,0.925317,0.016395
4446,borde_soacha,7a209b68,other,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,141.357970,57.951942,49.0,208.00000,...,0.691027,0.015662,0.196746,0.011014,0.182284,0.004717,0.959159,0.011052,0.936633,0.012371
4615,borde_soacha,7a360944,other,/content/drive/MyDrive/RoofMaterialProject/pat...,success,NaN,165.022770,55.789800,86.0,236.00000,...,0.768130,0.014489,0.300870,0.006488,0.293053,0.002204,0.964472,0.009912,0.944832,0.012934


In [13]:
import geopandas as gpd
import pandas as pd

ROOT = "/content/drive/MyDrive/RoofMaterialProject"

rural = gpd.read_file(
    ROOT + "/borde_rural/train-borde_rural.geojson"
)

soacha = gpd.read_file(
    ROOT + "/borde_soacha/train-borde_soacha.geojson"
)

labels = pd.concat([rural, soacha], ignore_index=True)

other_ids = set(other_samples["building_id"])

verified_check = labels[
    labels["id"].isin(other_ids)
][["id", "roof_material", "verified"]]

print("Matched records:", len(verified_check))
print("\nVerification distribution:")
print(verified_check["verified"].value_counts(dropna=False))

display(verified_check)

Matched records: 23

Verification distribution:
verified
True    23
Name: count, dtype: int64


,id,roof_material,verified
1008,7a391026,other,True
1244,7a3d7dfa,other,True
1573,7a1e6f46,other,True
1791,7a3dd502,other,True
1829,7a3f2984,other,True
2132,7a1dc41a,other,True
3124,7a255400,other,True
3523,7a343d44,other,True
4446,7a209b68,other,True
4615,7a360944,other,True


In [14]:
import matplotlib.pyplot as plt
from PIL import Image
import math
import os

other_samples = df_5class[
    df_5class["roof_material"] == "other"
].reset_index(drop=True)

n = len(other_samples)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(
    rows, cols,
    figsize=(16, rows * 4)
)

axes = axes.flatten()

for i, row in other_samples.iterrows():
    img = Image.open(row["patch_path"]).convert("RGB")

    axes[i].imshow(img)
    axes[i].set_title(
        f'{row["building_id"]}\n{row["region"]}',
        fontsize=9
    )
    axes[i].axis("off")

for j in range(n, len(axes)):
    axes[j].axis("off")

plt.suptitle("All 23 Verified Other-Class Roof Samples", fontsize=16)
plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.